![tower_bridge](tower_bridge.jpeg)

As the climate changes, predicting the weather becomes ever more important for businesses. Since the weather depends on a lot of different factors, you will want to run a lot of experiments to determine what the best approach is to predict the weather. In this project, you will run experiments for different regression models predicting the mean temperature, using a combination of `sklearn` and `MLflow`.

You will be working with data stored in `london_weather.csv`, which contains the following columns:
- **date** - recorded date of measurement - (**int**)
- **cloud_cover** - cloud cover measurement in oktas - (**float**)
- **sunshine** - sunshine measurement in hours (hrs) - (**float**)
- **global_radiation** - irradiance measurement in Watt per square meter (W/m2) - (**float**)
- **max_temp** - maximum temperature recorded in degrees Celsius (°C) - (**float**)
- **mean_temp** - mean temperature in degrees Celsius (°C) - (**float**)
- **min_temp** - minimum temperature recorded in degrees Celsius (°C) - (**float**)
- **precipitation** - precipitation measurement in millimeters (mm) - (**float**)
- **pressure** - pressure measurement in Pascals (Pa) - (**float**)
- **snow_depth** - snow depth measurement in centimeters (cm) - (**float**)

In [27]:
# Run this cell to import the modules you require
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

# Read in the data
weather = pd.read_csv("london_weather.csv",parse_dates=['date'])

# Extracting features from date field
weather['year']=weather['date'].dt.year
weather['month']=weather['date'].dt.month

#Dropping null target field records
weather=weather.dropna(subset=['mean_temp'])

# Splitting to train and test sets
X=weather.drop(columns=['mean_temp'])
y=weather['mean_temp']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,random_state=42)
print((X_train.shape,y_train.shape),(X_test.shape,y_test.shape))

#Building training pipeline
mlflow.set_experiment('london_mean_temp')
mlflow.autolog()


model_dict={'lin_reg':LinearRegression(),
            'dt_tree':DecisionTreeRegressor(),
            'rn_forest':RandomForestRegressor(random_state=42)}

col_trans=ColumnTransformer([('imputation',
                          SimpleImputer(strategy='median'),
                          ['cloud_cover', 'sunshine', 'global_radiation', 'precipitation', 'pressure', 'snow_depth','year', 'month'])])

for model_name,model in model_dict.items():

    with mlflow.start_run(run_name=model_name):

        mlflow.set_tag("model_type", model_name)
        train_pipeline=Pipeline([('transformer',col_trans),
                                 ('scaler',StandardScaler()),
                                 (f'{model_name}',model)])

        #train_pipeline.fit(X_train,y_train)

        param_grids = {
            "lin_reg": {},
            "dt_tree": {f"{model_name}__max_depth": range(1,11)},
            "rn_forest": {f"{model_name}__max_depth": range(1,11)},
        }   

        grid = GridSearchCV(train_pipeline, param_grids[model_name],scoring='neg_root_mean_squared_error', cv=5)

        grid.fit(X_train,y_train)
        
        mlflow.sklearn.log_model(sk_model=grid.best_estimator_,
                                 artifact_path=f"{model_name}",
                                 registered_model_name=f"best_{model_name}")

        rmse=mean_squared_error(y_test,grid.best_estimator_.predict(X_test),squared=False)

        mlflow.log_metric(f'best_rmse',rmse)

experiment_results=mlflow.search_runs()    




2026/09/27 15:49:46 INFO mlflow.tracking.fluent: Experiment with name 'london_mean_temp' does not exist. Creating a new experiment.
2026/09/27 15:49:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


((10713, 11), (10713,)) ((4592, 11), (4592,))


2026/09/27 15:49:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/09/27 15:49:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/09/27 15:49:53 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
Registered model 'best_lin_reg' already exists. Creating a new version of this model...
Created version '9' of model 'best_lin_reg'.
2026/09/27 15:50:07 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.
Registered model 'best_dt_tree' already exists. Creating a new version of this model...
Created version '8' of model 'best_dt_tree'.
2026/09/27 15:51:09 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.
Registered model 'best_rn_forest' already exists. Creating a new version of this model...
Created version '8' of model 'best_rn_forest'.


In [28]:
weather.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1


In [29]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15305 entries, 0 to 15340
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              15305 non-null  datetime64[ns]
 1   cloud_cover       15286 non-null  float64       
 2   sunshine          15305 non-null  float64       
 3   global_radiation  15286 non-null  float64       
 4   max_temp          15305 non-null  float64       
 5   mean_temp         15305 non-null  float64       
 6   min_temp          15305 non-null  float64       
 7   precipitation     15303 non-null  float64       
 8   pressure          15301 non-null  float64       
 9   snow_depth        13881 non-null  float64       
 10  year              15305 non-null  int64         
 11  month             15305 non-null  int64         
dtypes: datetime64[ns](1), float64(9), int64(2)
memory usage: 1.5 MB


In [30]:
#sns.pairplot(weather)

In [31]:
#weather.corr()

In [32]:
mlflow.search_runs(filter_string='metrics.best_rmse<3')

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.training_mean_absolute_error,metrics.training_mean_squared_error,metrics.best_rmse,metrics.training_r2_score,metrics.training_score,metrics.training_root_mean_squared_error,metrics.best_cv_score,params.verbose,params.error_score,params.cv,params.best_rn_forest__max_depth,params.param_grid,params.pre_dispatch,params.return_train_score,params.scoring,params.n_jobs,params.refit,params.estimator,params.best_dt_tree__max_depth,tags.mlflow.user,tags.model_type,tags.estimator_name,tags.estimator_class,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.log-model.history,tags.mlflow.source.name
0,c70246b09fc64038b992c37b2b257d06,426428429204478238,FINISHED,file:///work/files/workspace/mlruns/4264284292...,2026-09-27 15:50:14.484000+00:00,2026-09-27 15:51:17.173000+00:00,1.745588,4.806609,2.592049,0.853954,-2.192398,2.192398,-2.638230,0,nan,5,10,"{'rn_forest__max_depth': range(1, 11)}",2*n_jobs,False,neg_root_mean_squared_error,None,True,"Pipeline(steps=[('transformer',\n ...",None,repl,rn_forest,GridSearchCV,sklearn.model_selection._search.GridSearchCV,rn_forest,LOCAL,"[{""run_id"": ""c70246b09fc64038b992c37b2b257d06""...",/usr/lib/python3/dist-packages/python_kernel/k...
1,14d6f446bf6a4a6fa29080574c082c40,426428429204478238,FINISHED,file:///work/files/workspace/mlruns/4264284292...,2026-09-27 15:49:59.571000+00:00,2026-09-27 15:50:14.373000+00:00,2.124402,7.115163,2.788125,0.783811,-2.667426,2.667426,-2.835219,0,nan,5,None,"{'dt_tree__max_depth': range(1, 11)}",2*n_jobs,False,neg_root_mean_squared_error,None,True,"Pipeline(steps=[('transformer',\n ...",7,repl,dt_tree,GridSearchCV,sklearn.model_selection._search.GridSearchCV,dt_tree,LOCAL,"[{""run_id"": ""14d6f446bf6a4a6fa29080574c082c40""...",/usr/lib/python3/dist-packages/python_kernel/k...
